# 07 — Base Model with Chat Prefix

Tests whether using the Llama-3 chat template on the **base model** (no fine-tuning)
improves accuracy. This isolates the effect of prompt format vs. fine-tuning.

| Config | Chat Prefix | Rep Penalty | Expected Source |
|---|---|---|---|
| Base (raw prompt) | No | No | notebook 06 → 37.2% |
| **Base (chat format)** | **Yes** | **No** | **this notebook** |
| V1-fixed | Yes | 1.3 | notebook 06 → 51.8% |

In [1]:
# Same install as notebook 06
!pip install -q "datasets<3.0" transformers peft bitsandbytes accelerate torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 13.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [4]:
from huggingface_hub import login
login()

In [5]:
import torch, json, sqlite3, re
from tqdm.notebook import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

STOP_TOKEN_IDS = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|eot_id|>")]

# Same dataset load as notebook 06
ds = load_dataset("Salesforce/wikisql", trust_remote_code=True)
examples = list(ds['test'].select(range(500)))
print(f"Loaded {len(examples)} test examples")

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Generating test split:   0%|          | 0/15878 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8421 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/56355 [00:00<?, ? examples/s]

Loaded 500 test examples


In [6]:
# Generation config — no repetition_penalty (same as base in notebook 06)
GEN_CONFIG = dict(
    max_new_tokens=128,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=STOP_TOKEN_IDS,
)
print("Generation config:", GEN_CONFIG)

Generation config: {'max_new_tokens': 128, 'do_sample': False, 'pad_token_id': 128009, 'eos_token_id': [128009, 128009]}


In [7]:
# --- Post-processing (identical to notebook 06) ---

def fix_column_names(sql, columns):
    for col in sorted(columns, key=len, reverse=True):
        if col.lower() in sql.lower() and col not in sql:
            sql = re.sub(re.escape(col), col, sql, flags=re.IGNORECASE)
    return sql

def clean_wikisql(sql):
    sql = sql.strip()
    sql = re.sub(r'\bTABLE\b', 'table', sql)
    sql = re.sub(r'\s+', ' ', sql)
    match = re.match(r"(SELECT\s+.+?\s+FROM\s+table(?:\s+WHERE\s+.+)?)", sql, re.IGNORECASE)
    if match:
        sql = match.group(1)
    return sql.strip()

def extract_sql(text):
    text = text.strip()
    for prefix in ["SQL:", "sql:", "SQL :", "Answer:"]:
        if prefix in text:
            text = text.split(prefix, 1)[1].strip()
            break
    if "```" in text:
        match = re.search(r'```(?:sql)?\s*(.+?)```', text, re.DOTALL)
        if match:
            text = match.group(1).strip()
    lines = text.strip().split('\n')
    sql_lines = []
    for line in lines:
        if line.strip().upper().startswith(('SELECT', 'FROM', 'WHERE', 'AND', 'OR', 'ORDER', 'GROUP', 'HAVING', 'LIMIT', 'JOIN')):
            sql_lines.append(line.strip())
        elif sql_lines:
            break
    if sql_lines:
        return ' '.join(sql_lines)
    select_match = re.search(r'(SELECT\s+.+?)(?:\n\n|$)', text, re.IGNORECASE | re.DOTALL)
    if select_match:
        return select_match.group(1).strip()
    return text.strip()

print("Post-processing functions defined")

Post-processing functions defined


In [8]:
# --- Generation function with chat prefix enabled ---

CHAT_PREFIX = "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"

def generate_sql(model, question, columns, types, use_chat_prefix=True, gen_config=None):
    col_str = ", ".join([f"{c} ({t})" for c, t in zip(columns, types)])
    prompt = f"""Given the following table columns:\n{col_str}\n\nGenerate a SQL query for: {question}\n\nSQL:"""
    if use_chat_prefix:
        prompt = CHAT_PREFIX + prompt
    inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_config)
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

print("generate_sql() defined with use_chat_prefix=True")

generate_sql() defined with use_chat_prefix=True


In [9]:
# --- Evaluation (identical to notebook 06) ---

def execute_sql(sql, columns, table_data):
    conn = sqlite3.connect(':memory:')
    cursor = conn.cursor()
    col_defs = ', '.join([f'"{c}" TEXT' for c in columns])
    cursor.execute(f'CREATE TABLE "table" ({col_defs})')
    if table_data:
        placeholders = ', '.join(['?' for _ in columns])
        for row in table_data:
            cursor.execute(f'INSERT INTO "table" VALUES ({placeholders})', [str(v) for v in row])
    try:
        cursor.execute(sql)
        result = cursor.fetchall()
    except Exception as e:
        result = f"ERROR: {e}"
    conn.close()
    return result

def build_gold_sql(example):
    sql_dict = example['sql']
    columns = example['table']['header']
    agg_ops = ['', 'MAX', 'MIN', 'COUNT', 'SUM', 'AVG']
    cond_ops = ['=', '>', '<']
    sel_col = columns[sql_dict['sel']]
    agg = agg_ops[sql_dict['agg']]
    if agg:
        select_clause = f"SELECT {agg}({sel_col})"
    else:
        select_clause = f"SELECT {sel_col}"
    where_clause = ""
    conds = sql_dict['conds']
    if conds['column_index']:
        conditions = []
        for i in range(len(conds['column_index'])):
            col = columns[conds['column_index'][i]]
            op = cond_ops[conds['operator_index'][i]]
            val = conds['condition'][i]
            conditions.append(f"{col} {op} {val}")
        where_clause = " WHERE " + " AND ".join(conditions)
    return f"{select_clause} FROM table{where_clause}"

print("Evaluation functions defined")

Evaluation functions defined


In [10]:
# --- Load base model (same bnb_config as notebook 06) ---

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()
print(f"Base model loaded. Memory: {torch.cuda.memory_allocated()/1e9:.1f} GB")

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Base model loaded. Memory: 6.1 GB


In [11]:
# --- Sanity check: 3 examples ---

print("=== Base Model + Chat Prefix Sanity Check ===")
for i in range(3):
    ex = examples[i]
    pred = generate_sql(model, ex['question'], ex['table']['header'],
                        ex['table']['types'], use_chat_prefix=True, gen_config=GEN_CONFIG)
    pred = extract_sql(pred)
    pred = fix_column_names(pred, ex['table']['header'])
    gold = build_gold_sql(ex)
    print(f"\nQ:    {ex['question']}")
    print(f"Pred: {pred}")
    print(f"Gold: {gold}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== Base Model + Chat Prefix Sanity Check ===

Q:    What is terrence ross' nationality
Pred: 
Gold: SELECT Nationality FROM table WHERE Player = Terrence Ross

Q:    What clu was in toronto 1995-96
Pred: SELECT School/Club Team FROM table_name WHERE Years in Toronto = '1995-96';
Gold: SELECT School/Club Team FROM table WHERE Years in Toronto = 1995-96

Q:    which club was in toronto 2003-06
Pred: SELECT School/Club Team FROM table_name WHERE Years in Toronto BETWEEN '2003' AND '2006'
Gold: SELECT School/Club Team FROM table WHERE Years in Toronto = 2003-06


In [12]:
# --- Run 500 predictions ---

predictions = []
for i in tqdm(range(len(examples)), desc="Base+chat"):
    ex = examples[i]
    raw = generate_sql(model, ex['question'], ex['table']['header'],
                       ex['table']['types'], use_chat_prefix=True, gen_config=GEN_CONFIG)
    sql = extract_sql(raw)
    sql = fix_column_names(sql, ex['table']['header'])
    gold = build_gold_sql(ex)

    pred_result = execute_sql(sql, ex['table']['header'], ex['table']['rows'])
    gold_result = execute_sql(gold, ex['table']['header'], ex['table']['rows'])

    predictions.append({
        'index': i,
        'question': ex['question'],
        'predicted_sql': sql,
        'gold_sql': gold,
        'predicted_result': str(pred_result),
        'gold_result': str(gold_result),
        'correct': str(pred_result) == str(gold_result),
        'is_error': str(pred_result).startswith('ERROR'),
    })

print(f"Generated {len(predictions)} predictions")

Base+chat:   0%|          | 0/500 [00:00<?, ?it/s]

Generated 500 predictions


In [13]:
# --- Results ---

correct = sum(1 for p in predictions if p['correct'])
errors = sum(1 for p in predictions if p['is_error'])
wrong = len(predictions) - correct - errors

print(f"Base model + chat prefix results:")
print(f"  Execution accuracy: {correct/len(predictions)*100:.1f}%")
print(f"  Syntax error rate:  {errors/len(predictions)*100:.1f}%")
print(f"  Wrong result rate:  {wrong/len(predictions)*100:.1f}%")

Base model + chat prefix results:
  Execution accuracy: 1.8%
  Syntax error rate:  49.0%
  Wrong result rate:  49.2%


In [14]:
# --- Save and download ---

results = {
    'model': 'Meta-Llama-3-8B-Instruct (base, no fine-tuning)',
    'prompt_format': 'chat_prefix',
    'generation_config': {k: str(v) for k, v in GEN_CONFIG.items()},
    'num_examples': len(predictions),
    'execution_accuracy': correct / len(predictions),
    'syntax_error_rate': errors / len(predictions),
    'wrong_result_rate': wrong / len(predictions),
    'predictions': predictions,
}

with open('base_model_chat_prefix.json', 'w') as f:
    json.dump(results, f, indent=2)

from google.colab import files
files.download('base_model_chat_prefix.json')
print("Saved and downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved and downloaded!


In [15]:
# --- Final 3-way comparison table ---

base_chat_acc = correct / len(predictions) * 100
base_chat_err = errors / len(predictions) * 100
base_chat_wrong = wrong / len(predictions) * 100

print("="*70)
print("FULL COMPARISON")
print("="*70)
print(f"{'Config':<35} {'Exec Acc':>10} {'Syntax Err':>12} {'Wrong':>8}")
print("-"*70)
print(f"{'Base (raw prompt, no RP)':<35} {'37.2%':>10} {'21.8%':>12} {'41.0%':>8}")
print(f"{'Base (chat prefix, no RP)':<35} {f'{base_chat_acc:.1f}%':>10} {f'{base_chat_err:.1f}%':>12} {f'{base_chat_wrong:.1f}%':>8}")
print(f"{'V1-fixed (chat prefix, RP=1.3)':<35} {'51.8%':>10} {'5.4%':>12} {'42.8%':>8}")
print("-"*70)
print(f"\nChat prefix effect on base:        37.2% → {base_chat_acc:.1f}% ({base_chat_acc-37.2:+.1f}%)")
print(f"Fine-tuning effect (vs base+chat): {base_chat_acc:.1f}% → 51.8% ({51.8-base_chat_acc:+.1f}%)")

FULL COMPARISON
Config                                Exec Acc   Syntax Err    Wrong
----------------------------------------------------------------------
Base (raw prompt, no RP)                 37.2%        21.8%    41.0%
Base (chat prefix, no RP)                 1.8%        49.0%    49.2%
V1-fixed (chat prefix, RP=1.3)           51.8%         5.4%    42.8%
----------------------------------------------------------------------

Chat prefix effect on base:        37.2% → 1.8% (-35.4%)
Fine-tuning effect (vs base+chat): 1.8% → 51.8% (+50.0%)
